In [1]:
import pandas as pd
import numpy as np

In [3]:
import warnings;
warnings.simplefilter('ignore')

In [5]:
business_df = pd.read_json('yelp_academic_dataset_business.json', lines = True)

In [7]:
business_df.shape

(150346, 14)

In [9]:
reviews_df = pd.read_parquet('review_subset.parquet')

In [11]:
reviews_df.shape

(3840621, 9)

In [13]:
# checkin_df = pd.read_json('yelp_academic_dataset_checkin.json', lines = True)

In [15]:
# checkin_df.shape

In [17]:
# checkin_df.columns

In [19]:
# tip_df = pd.read_json('yelp_academic_dataset_tip.json', lines = True)

In [21]:
# tip_df.shape

In [662]:
user_df = pd.read_json('yelp_academic_dataset_user.json', lines = True)

In [664]:
user_df.shape

(1987897, 22)

In [674]:
user_df.columns

Index(['user_id', 'name', 'review_count', 'yelping_since', 'useful', 'funny',
       'cool', 'elite', 'friends', 'fans', 'average_stars', 'compliment_hot',
       'compliment_more', 'compliment_profile', 'compliment_cute',
       'compliment_list', 'compliment_note', 'compliment_plain',
       'compliment_cool', 'compliment_funny', 'compliment_writer',
       'compliment_photos'],
      dtype='object')

In [27]:
business_df.columns

Index(['business_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'stars', 'review_count', 'is_open',
       'attributes', 'categories', 'hours'],
      dtype='object')

In [29]:
reviews_df.columns

Index(['text', 'cool', 'stars', 'date', 'funny', 'review_id', 'useful',
       'business_id', 'user_id'],
      dtype='object')

In [31]:
business_df.head(3)

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."


In [33]:
def split_and_explode(row):
    categories = row['categories']
    if categories is not None and categories.strip() != '':
        categories = categories.split(', ')
        return pd.Series({'business_id': row['business_id'], 'business_category': categories})
    else:
        return pd.Series({'business_id': row['business_id'], 'business_category': []})

# Apply the function to split and explode categories
business_categories = business_df.apply(split_and_explode, axis=1)

In [35]:
# Explode the list of business categories
business_categories = business_categories.explode('business_category')

In [39]:
restaurant_business_df = business_categories[business_categories['business_category'] == 'Restaurants']

In [41]:
restaurant_business_df.shape

(52268, 2)

In [43]:
restaurant_business_merge = pd.merge(restaurant_business_df, business_df, left_on='business_id', right_on='business_id', how='left')

In [45]:
restaurant_business_merge.shape

(52268, 15)

In [47]:
restaurant_business_merge.columns

Index(['business_id', 'business_category', 'name', 'address', 'city', 'state',
       'postal_code', 'latitude', 'longitude', 'stars', 'review_count',
       'is_open', 'attributes', 'categories', 'hours'],
      dtype='object')

In [49]:
restaurant_business_merge.head(3)

,business_id,business_category,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,MTSW4McQd7CbVtyjqoe9mw,Restaurants,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
1,CF33F8-E6oudUQ46HnavjQ,Restaurants,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,1,"{'BusinessParking': 'None', 'BusinessAcceptsCr...","Burgers, Fast Food, Sandwiches, Food, Ice Crea...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-22:0', '..."
2,k0hlBqXX-Bt0vf1op7Jr1w,Restaurants,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,0,"{'Caters': 'True', 'Alcohol': 'u'full_bar'', '...","Pubs, Restaurants, Italian, Bars, American (Tr...",None


In [51]:
# rows = []

# # Define the days of the week
# days_of_week = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# # Iterate over rows in the DataFrame
# for _, row in business_df.iterrows():
#     business_id = row['business_id']
#     hours = row['hours']
    
#     if hours is not None:
#         for day_of_week in days_of_week:
#             time_range = hours.get(day_of_week)
#             if time_range:
#                 open_time, close_time = time_range.split('-')
#                 rows.append({
#                     'business_id': business_id,
#                     'day_of_week': day_of_week,
#                     'open_time': open_time,
#                     'close_time': close_time
#                 })

# # Create a DataFrame from the list of rows
# business_hours = pd.DataFrame(rows)

In [53]:
# business_hours

In [55]:
business_attributes_to_col = business_df['attributes'].apply(pd.Series)

In [57]:
business_attributes_to_col_merge1 = pd.concat([business_df, business_attributes_to_col], axis = 1)

In [59]:
import ast

parking_df = business_attributes_to_col_merge1['BusinessParking'].apply(
    lambda x: pd.Series(ast.literal_eval(x)) if pd.notna(x) else pd.Series({'garage': None, 'street': None, 'validated': None, 'lot': None, 'valet': None}))

In [267]:
business_df_final = pd.concat([business_attributes_to_col_merge1, parking_df], axis=1)

In [269]:
business_df_final.shape

(150346, 58)

In [271]:
business_df_final.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,...,HairSpecializesIn,Open24Hours,RestaurantsCounterService,AgesAllowed,DietaryRestrictions,garage,street,validated,lot,valet
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,...,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,...,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,...,NaN,NaN,NaN,NaN,NaN,False,False,False,True,False
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,...,NaN,NaN,NaN,NaN,NaN,False,True,False,False,False
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,...,NaN,NaN,NaN,NaN,NaN,None,None,None,True,False


In [540]:
business_restaurant_final = pd.merge(restaurant_business_merge, business_df_final, left_on='business_id', right_on='business_id', how='left')

In [542]:
business_restaurant_final.shape

(52268, 72)

In [544]:
business_restaurant_final.columns

Index(['business_id', 'business_category', 'name_x', 'address_x', 'city_x',
       'state_x', 'postal_code_x', 'latitude_x', 'longitude_x', 'stars_x',
       'review_count_x', 'is_open_x', 'attributes_x', 'categories_x',
       'hours_x', 'name_y', 'address_y', 'city_y', 'state_y', 'postal_code_y',
       'latitude_y', 'longitude_y', 'stars_y', 'review_count_y', 'is_open_y',
       'attributes_y', 'categories_y', 'hours_y', 'ByAppointmentOnly',
       'BusinessAcceptsCreditCards', 'BikeParking', 'RestaurantsPriceRange2',
       'CoatCheck', 'RestaurantsTakeOut', 'RestaurantsDelivery', 'Caters',
       'WiFi', 'BusinessParking', 'WheelchairAccessible', 'HappyHour',
       'OutdoorSeating', 'HasTV', 'RestaurantsReservations', 'DogsAllowed',
       'Alcohol', 'GoodForKids', 'RestaurantsAttire', 'Ambience',
       'RestaurantsTableService', 'RestaurantsGoodForGroups', 'DriveThru',
       'NoiseLevel', 'GoodForMeal', 'BusinessAcceptsBitcoin', 'Smoking',
       'Music', 'GoodForDancing', 'Ac

In [546]:
business_restaurant_final.head()

,business_id,business_category,name_x,address_x,city_x,state_x,postal_code_x,latitude_x,longitude_x,stars_x,...,HairSpecializesIn,Open24Hours,RestaurantsCounterService,AgesAllowed,DietaryRestrictions,garage,street,validated,lot,valet
0,MTSW4McQd7CbVtyjqoe9mw,Restaurants,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,...,NaN,NaN,NaN,NaN,NaN,False,True,False,False,False
1,CF33F8-E6oudUQ46HnavjQ,Restaurants,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,k0hlBqXX-Bt0vf1op7Jr1w,Restaurants,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,...,NaN,NaN,NaN,NaN,NaN,False,False,False,True,False
3,bBDDEgkFA1Otx9Lfe7BZUQ,Restaurants,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,...,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False
4,eEOYSgkmpB90uNA7lDOMRA,Restaurants,Vietnamese Food Truck,,Tampa Bay,FL,33602,27.955269,-82.456320,4.0,...,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False


In [548]:
def replace_none_with_false(df, columns):
    """Replaces 'None' string values with 'False' in specified columns."""
    for col in columns:
        df[col] = np.where(df[col] == 'None', 'False', df[col])
    return df

In [550]:
columns_to_replace = ['WheelchairAccessible', 'Corkage', 'DriveThru', 'GoodForKids', 'RestaurantsGoodForGroups', 'RestaurantsTableService', 'BusinessAcceptsCreditCards', 'DogsAllowed','RestaurantsReservations', 'HappyHour', 'OutdoorSeating','HasTV','Caters', 'RestaurantsDelivery', 'RestaurantsTakeOut', 'CoatCheck', 'BikeParking']

In [552]:
business_restaurant_final = replace_none_with_false(business_restaurant_final, columns_to_replace)

In [554]:
def classify_music(music):
    """Classifies music as 'Good' if any value is True, otherwise 'Bad'."""
    if music is None or (isinstance(music, float) and np.isnan(music)):  
        return 'False'  # Assume missing values mean 'Bad' music
    
    if isinstance(music, str):  
        try:
            music = ast.literal_eval(music)  # Convert string representation to dictionary
        except (SyntaxError, ValueError):
            return 'False'  # If conversion fails, assume 'Bad'

    if isinstance(music, dict):
        return 'True' if any(music.values()) else 'False'
    
    return 'False'  # Default case

In [556]:
business_restaurant_final['MusicCategory'] = business_restaurant_final['Music'].apply(classify_music)

In [557]:
def classify_ambience(ambience):
    """Classifies ambience as 'Good' if any value is True, otherwise 'Bad'."""
    if ambience is None or (isinstance(ambience, float) and np.isnan(ambience)):  
        return 'Bad'  # Assume missing values mean 'Bad' ambience
    
    if isinstance(ambience, str):  
        try:
            ambience = ast.literal_eval(ambience)  # Convert string representation to dictionary
        except (SyntaxError, ValueError):
            return 'Bad'  # If conversion fails, assume 'Bad'

    if isinstance(ambience, dict):
        return 'Good' if any(ambience.values()) else 'Bad'
    
    return 'Bad'  # Default case

In [560]:
business_restaurant_final['AmbienceCategory'] = business_restaurant_final['Ambience'].apply(classify_ambience)

In [561]:
wifi_mapping = {
    "u'free'": "True", 
    "'free'": "True",
    "u'no'": "False", 
    "'no'": "False",
    "u'paid'": "True", 
    "'paid'": "True",
    "None" : "False"
}

In [562]:
business_restaurant_final['WiFi'] = business_restaurant_final['WiFi'].map(lambda x: wifi_mapping.get(x, x))

In [563]:
business_restaurant_final['Alcohol'] = business_restaurant_final['Alcohol'].replace(
    { 
        "u'full_bar'": True, "'full_bar'": True, 
        "u'beer_and_wine'": True, "'beer_and_wine'": True, 
        "u'none'": False, "'none'": False, 
        'None': False
    }
)

In [564]:
business_restaurant_final['NoiseLevel'] = business_restaurant_final['NoiseLevel'].replace(
    {
        "u'quiet'": "quiet", "'quiet'": "quiet",
        "u'loud'": "loud", "'loud'": "loud",
        "u'very_loud'": "loud", "'very_loud'": "loud",
        "u'average'": "average", "'average'": "average",
        'None': "average"
    }
)

In [565]:
business_restaurant_final['Smoking'] = business_restaurant_final['Smoking'].replace(
    {
        "u'no'": "no", "'no'": "no",
        "u'yes'": "yes", "'yes'": "yes",
        "u'outdoor'": "yes", "'outdoor'": "yes",
        'None': "no"
    }
)

In [570]:
business_restaurant_final['parking'] = business_restaurant_final[['garage', 'street', 'validated', 'lot', 'valet']].any(axis=1)

In [574]:
business_restaurant_final['parking_final'] = business_restaurant_final[['parking', 'BikeParking']].any(axis=1)

In [576]:
columns_to_drop = ['name_y','attributes_x', 'address_x', 'postal_code_x', 'latitude_x', 'longitude_x','parking', 'BikeParking','categories_x', 'BusinessAcceptsBitcoin', 'address_y', 'city_y', 'state_y', 'postal_code_y', 'garage', 'street', 'validated', 'lot', 'valet',
       'latitude_y', 'longitude_y', 'stars_y', 'review_count_y', 'is_open_y',
       'attributes_y', 'categories_y', 'hours_y', 'RestaurantsPriceRange2', 'BusinessParking', 'Ambience', 'GoodForMeal', 'Music', 'GoodForDancing', 'AcceptsInsurance', 'BestNights', 'BYOB', 'BYOBCorkage', 'HairSpecializesIn', 'Open24Hours', 'RestaurantsCounterService','AgesAllowed','DietaryRestrictions','RestaurantsAttire']
business_restaurant_final = business_restaurant_final.drop(columns=columns_to_drop)

In [578]:
business_restaurant_final.columns

Index(['business_id', 'business_category', 'name_x', 'city_x', 'state_x',
       'stars_x', 'review_count_x', 'is_open_x', 'hours_x',
       'ByAppointmentOnly', 'BusinessAcceptsCreditCards', 'CoatCheck',
       'RestaurantsTakeOut', 'RestaurantsDelivery', 'Caters', 'WiFi',
       'WheelchairAccessible', 'HappyHour', 'OutdoorSeating', 'HasTV',
       'RestaurantsReservations', 'DogsAllowed', 'Alcohol', 'GoodForKids',
       'RestaurantsTableService', 'RestaurantsGoodForGroups', 'DriveThru',
       'NoiseLevel', 'Smoking', 'Corkage', 'MusicCategory', 'AmbienceCategory',
       'parking_final'],
      dtype='object')

In [580]:
for col in business_restaurant_final.columns:
    mode_value = business_restaurant_final[col].mode()[0]  # Get the most frequent value
    business_restaurant_final[col].fillna(mode_value, inplace=True)

In [582]:
business_restaurant_final = business_restaurant_final.rename(columns={'name_x': 'business_name', 'stars_x': 'business_star_rating', 'hours_x': 'business_working_hours'})

In [584]:
reviews_df.head()

,text,reviews_received_cool,stars,date,reviews_received_funny,review_id,reviews_received_useful,business_id,user_id
0,ChopHouse\n\nWith everything going on in the w...,4,5.0,2021-07-08 02:31:34+00:00,1,iMaxU340B3Gz43hAsaDmRw,6,iC9Gis3-VspIr8Ox3e2beA,p951o8o4e3cHC1DSyiKP7Q
1,Another spot that just doesn't disappoint! The...,0,5.0,2019-05-16 03:36:13+00:00,0,hl4PnDzKGaF8xTwStIfDXw,0,M-6AXSgmDMYcWsF442mzzA,gFFMMMlnoqt-3YkfhaaI9Q
2,"As a same day wedding coordinator, this shop i...",0,5.0,2017-02-07 19:22:29+00:00,0,Afs4ChtZB4yf6yapfoo0Hg,0,Rnvl_hNexRJvC9zwVzPOdw,xShyBuTNL2mFZyvkL0vtPQ
3,This was possibly the best Mexican meal I've e...,0,5.0,2018-09-16 14:06:17+00:00,0,wZgsjr1hA5HJ189Bx8FPfQ,0,1Ly-Njk6U0kEq3TFnCgeWw,MTPka8o3xwDpEMMGQBtg5g
4,"Had a head unit, amplifier and a set of compon...",0,5.0,2020-09-15 21:48:29+00:00,0,Ac2n_YmBBeIEcRrOYOWZOg,0,h_vx0fR8lnwsrJASl_S8Aw,AgAqKRshVVuQQdxVw9zMxA


In [586]:
reviews_df = reviews_df.rename(columns={'stars': 'reviews_star_rating', 'useful': 'reviews_received_useful', 'funny': 'reviews_received_funny', 'cool': 'reviews_received_cool'})

In [588]:
reviews_df.head()

,text,reviews_received_cool,reviews_star_rating,date,reviews_received_funny,review_id,reviews_received_useful,business_id,user_id
0,ChopHouse\n\nWith everything going on in the w...,4,5.0,2021-07-08 02:31:34+00:00,1,iMaxU340B3Gz43hAsaDmRw,6,iC9Gis3-VspIr8Ox3e2beA,p951o8o4e3cHC1DSyiKP7Q
1,Another spot that just doesn't disappoint! The...,0,5.0,2019-05-16 03:36:13+00:00,0,hl4PnDzKGaF8xTwStIfDXw,0,M-6AXSgmDMYcWsF442mzzA,gFFMMMlnoqt-3YkfhaaI9Q
2,"As a same day wedding coordinator, this shop i...",0,5.0,2017-02-07 19:22:29+00:00,0,Afs4ChtZB4yf6yapfoo0Hg,0,Rnvl_hNexRJvC9zwVzPOdw,xShyBuTNL2mFZyvkL0vtPQ
3,This was possibly the best Mexican meal I've e...,0,5.0,2018-09-16 14:06:17+00:00,0,wZgsjr1hA5HJ189Bx8FPfQ,0,1Ly-Njk6U0kEq3TFnCgeWw,MTPka8o3xwDpEMMGQBtg5g
4,"Had a head unit, amplifier and a set of compon...",0,5.0,2020-09-15 21:48:29+00:00,0,Ac2n_YmBBeIEcRrOYOWZOg,0,h_vx0fR8lnwsrJASl_S8Aw,AgAqKRshVVuQQdxVw9zMxA


In [594]:
business_reviews_merged = pd.merge(business_restaurant_final, reviews_df, on='business_id', how='left')

In [606]:
business_reviews_merged['is_open_x'] = business_reviews_merged['is_open_x'].replace(
    {
        1: "yes", 0: "no"
    }
)

In [658]:
business_reviews_merged['city_x'] = business_reviews_merged['city_x'].replace(
    {
        'St. Louis': "Saint Louis", 'St Louis': "Saint Louis", 'Clearwater Beach': 'Clearwater', 'St. Petersburg': 'Saint Petersburg', 'Abington Township': 'Abington',
        'Ashland City':'Ashland','St Petersburg': 'Saint Petersburg','St Pete Beach':'St. Pete Beach'
    }
)

In [694]:
unique_user_id = list(business_reviews_merged['user_id'].unique())

In [698]:
user_subset = user_df[user_df['user_id'].isin(unique_user_id)]

In [702]:
user_subset.to_parquet('user_subset.parquet')

In [704]:
business_reviews_merged.to_parquet('business_reviews_merged.parquet')